In [1]:
import polars as pl
import pandas as pd
import pyranges as pr

/opt/modules/i12g/anaconda/envs/sl-ukg/lib/python3.12/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Polars only code

In [2]:
gtf_path = "/s/project/deeprvat/ukb_gym/gencode/gencode.v49.annotation.gtf.gz"

gtf_pl = pl.read_csv(
    gtf_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=["Chromosome", "source", "Feature", "Start", "End", "score", "Strand", "frame", "attributes"]
)

gtf_pl = gtf_pl.with_row_index("row_nr")
gtf_pl

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes
u32,str,str,str,i64,i64,str,str,str,str
0,"""chr1""","""HAVANA""","""gene""",11121,24894,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; g…"
1,"""chr1""","""HAVANA""","""transcript""",11121,14413,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…"
2,"""chr1""","""HAVANA""","""exon""",11121,11211,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…"
3,"""chr1""","""HAVANA""","""exon""",12010,12227,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…"
4,"""chr1""","""HAVANA""","""exon""",12613,12721,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…"
…,…,…,…,…,…,…,…,…,…
7750149,"""chrM""","""ENSEMBL""","""transcript""",15888,15953,""".""","""+""",""".""","""gene_id ""ENSG00000210195.2""; t…"
7750150,"""chrM""","""ENSEMBL""","""exon""",15888,15953,""".""","""+""",""".""","""gene_id ""ENSG00000210195.2""; t…"
7750151,"""chrM""","""ENSEMBL""","""gene""",15956,16023,""".""","""-""",""".""","""gene_id ""ENSG00000210196.2""; g…"


In [3]:
atts = (
    gtf_pl.select("row_nr", "attributes")
    
    .with_columns(
        attrs_list=pl.col("attributes").str.split("; ")
    )
    .explode("attrs_list")
    
    .with_columns(
        pl.col("attrs_list")
        .str.split_exact(" ", 1)
        .struct.rename_fields(["attribute", "value"])
        .alias("fields")
    ).unnest("fields")

    .with_columns(
        pl.col("value").str.strip_chars('"')
    )

    .pivot(
        index="row_nr",
        on="attribute",
        values="value",
        aggregate_function=pl.element().implode()
    )

    .with_columns(
        pl.col(pl.List).list.join(", ").str.strip_chars(";").replace("", None)
    )

    .with_columns(
        level = pl.col('level').cast(pl.Int32),
        exon_number = pl.col('exon_number').cast(pl.Int32),
    )
)

atts

row_nr,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
u32,str,str,str,i32,str,str,str,str,i32,str,str,str,str,str,str,str,str,str
0,"""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""overlaps_pseudogene""""",null,null,null,null,null,null,null,null,null,null,null,null,null
1,"""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",null,null,null,null,null,null,null,null,null,null
2,"""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",1,"""ENSE00004248723.1""",null,null,null,null,null,null,null,null
3,"""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",2,"""ENSE00004248735.1""",null,null,null,null,null,null,null,null
4,"""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",3,"""ENSE00003582793.1""",null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7750149,"""ENSG00000210195.2""","""Mt_tRNA""","""MT-TT""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000387460.2""","""Mt_tRNA""","""MT-TT-201""",null,null,"""NA""",null,"""HGNC:7499""",null,null,null,null,null
7750150,"""ENSG00000210195.2""","""Mt_tRNA""","""MT-TT""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000387460.2""","""Mt_tRNA""","""MT-TT-201""",1,"""ENSE00001544475.2""","""NA""",null,"""HGNC:7499""",null,null,null,null,null
7750151,"""ENSG00000210196.2""","""Mt_tRNA""","""MT-TP""",3,null,null,null,null,null,null,null,null,"""HGNC:7494""""",null,null,null,null,null


In [4]:
gtf_exp = gtf_pl.join(atts, on='row_nr')
gtf_exp

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
u32,str,str,str,i64,i64,str,str,str,str,str,str,str,i32,str,str,str,str,i32,str,str,str,str,str,str,str,str,str
0,"""chr1""","""HAVANA""","""gene""",11121,24894,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; g…","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""overlaps_pseudogene""""",null,null,null,null,null,null,null,null,null,null,null,null,null
1,"""chr1""","""HAVANA""","""transcript""",11121,14413,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",null,null,null,null,null,null,null,null,null,null
2,"""chr1""","""HAVANA""","""exon""",11121,11211,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",1,"""ENSE00004248723.1""",null,null,null,null,null,null,null,null
3,"""chr1""","""HAVANA""","""exon""",12010,12227,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",2,"""ENSE00004248735.1""",null,null,null,null,null,null,null,null
4,"""chr1""","""HAVANA""","""exon""",12613,12721,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; t…","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""",2,"""TAGENE""""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",3,"""ENSE00003582793.1""",null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7750149,"""chrM""","""ENSEMBL""","""transcript""",15888,15953,""".""","""+""",""".""","""gene_id ""ENSG00000210195.2""; t…","""ENSG00000210195.2""","""Mt_tRNA""","""MT-TT""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000387460.2""","""Mt_tRNA""","""MT-TT-201""",null,null,"""NA""",null,"""HGNC:7499""",null,null,null,null,null
7750150,"""chrM""","""ENSEMBL""","""exon""",15888,15953,""".""","""+""",""".""","""gene_id ""ENSG00000210195.2""; t…","""ENSG00000210195.2""","""Mt_tRNA""","""MT-TT""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000387460.2""","""Mt_tRNA""","""MT-TT-201""",1,"""ENSE00001544475.2""","""NA""",null,"""HGNC:7499""",null,null,null,null,null
7750151,"""chrM""","""ENSEMBL""","""gene""",15956,16023,""".""","""-""",""".""","""gene_id ""ENSG00000210196.2""; g…","""ENSG00000210196.2""","""Mt_tRNA""","""MT-TP""",3,null,null,null,null,null,null,null,null,"""HGNC:7494""""",null,null,null,null,null


In [5]:
a = gtf_exp['gene_type'].value_counts(sort=True)
a

gene_type,count
str,u32
"""protein_coding""",6731674
"""lncRNA""",932062
"""processed_pseudogene""",29949
"""transcribed_unprocessed_pseudo…",12212
"""unprocessed_pseudogene""",8511
…,…
"""TR_J_pseudogene""",12
"""IG_J_pseudogene""",9
"""translated_processed_pseudogen…",8


In [6]:
mane_exons = (
    gtf_exp
    .filter(
        (pl.col('gene_type') == 'protein_coding') |
        (pl.col('gene_type').str.contains('pseudogene')) |
        (pl.col('Feature') == 'CDS')
    )
    .filter(pl.col('Feature') != 'transcript')
    .filter(pl.col('tag').str.contains('MANE_Select'))
)

mane_exons

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
u32,str,str,str,i64,i64,str,str,str,str,str,str,str,i32,str,str,str,str,i32,str,str,str,str,str,str,str,str,str
2488,"""chr1""","""HAVANA""","""exon""",65419,65433,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""protein_coding""","""OR4F5""",2,"""RNA_Seq_supported_partial, bas…","""ENST00000641515.2""","""protein_coding""","""OR4F5-201""",1,"""ENSE00003812156.1""",null,"""OTTHUMT00000003223.4""""","""HGNC:14825""","""OTTHUMG00000001094.4""",null,"""ENSP00000493376.2""","""CCDS30547.2""",null
2489,"""chr1""","""HAVANA""","""exon""",65520,65573,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""protein_coding""","""OR4F5""",2,"""RNA_Seq_supported_partial, bas…","""ENST00000641515.2""","""protein_coding""","""OR4F5-201""",2,"""ENSE00003813641.1""",null,"""OTTHUMT00000003223.4""""","""HGNC:14825""","""OTTHUMG00000001094.4""",null,"""ENSP00000493376.2""","""CCDS30547.2""",null
2490,"""chr1""","""HAVANA""","""CDS""",65565,65573,""".""","""+""","""0""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""protein_coding""","""OR4F5""",2,"""RNA_Seq_supported_partial, bas…","""ENST00000641515.2""","""protein_coding""","""OR4F5-201""",2,"""ENSE00003813641.1""",null,"""OTTHUMT00000003223.4""""","""HGNC:14825""","""OTTHUMG00000001094.4""",null,"""ENSP00000493376.2""","""CCDS30547.2""",null
2491,"""chr1""","""HAVANA""","""start_codon""",65565,65567,""".""","""+""","""0""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""protein_coding""","""OR4F5""",2,"""RNA_Seq_supported_partial, bas…","""ENST00000641515.2""","""protein_coding""","""OR4F5-201""",2,"""ENSE00003813641.1""",null,"""OTTHUMT00000003223.4""""","""HGNC:14825""","""OTTHUMG00000001094.4""",null,"""ENSP00000493376.2""","""CCDS30547.2""",null
2492,"""chr1""","""HAVANA""","""exon""",69037,71585,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""protein_coding""","""OR4F5""",2,"""RNA_Seq_supported_partial, bas…","""ENST00000641515.2""","""protein_coding""","""OR4F5-201""",3,"""ENSE00003813949.1""",null,"""OTTHUMT00000003223.4""""","""HGNC:14825""","""OTTHUMG00000001094.4""",null,"""ENSP00000493376.2""","""CCDS30547.2""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7749027,"""chrY""","""HAVANA""","""exon""",25624455,25624902,""".""","""+""",""".""","""gene_id ""ENSG00000172288.8""; t…","""ENSG00000172288.8""","""protein_coding""","""CDY1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000306609.5""","""protein_coding""","""CDY1-201""",2,"""ENSE00001666303.1""","""1""","""OTTHUMT00000105031.1""""","""HGNC:1809""","""OTTHUMG00000045274.2""",null,"""ENSP00000302968.4""","""CCDS14801.1""",null
7749028,"""chrY""","""HAVANA""","""CDS""",25624455,25624524,""".""","""+""","""1""","""gene_id ""ENSG00000172288.8""; t…","""ENSG00000172288.8""","""protein_coding""","""CDY1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000306609.5""","""protein_coding""","""CDY1-201""",2,"""ENSE00001666303.1""","""1""","""OTTHUMT00000105031.1""""","""HGNC:1809""","""OTTHUMG00000045274.2""",null,"""ENSP00000302968.4""","""CCDS14801.1""",null
7749029,"""chrY""","""HAVANA""","""stop_codon""",25624525,25624527,""".""","""+""","""0""","""gene_id ""ENSG00000172288.8""; t…","""ENSG00000172288.8""","""protein_coding""","""CDY1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000306609.5""","""protein_coding""","""CDY1-201""",2,"""ENSE00001666303.1""","""1""","""OTTHUMT00000105031.1""""","""HGNC:1809""","""OTTHUMG00000045274.2""",null,"""ENSP00000302968.4""","""CCDS14801.1""",null


In [7]:
# One transcript per gene
mane_exons[['gene_id', 'gene_name', 'transcript_id']].unique()['transcript_id'].value_counts(sort=True)

transcript_id,count
str,u32
"""ENST00000644903.1""",1
"""ENST00000646101.2""",1
"""ENST00000332853.9""",1
"""ENST00000629482.3""",1
"""ENST00000639785.2""",1
…,…
"""ENST00000396403.9""",1
"""ENST00000302917.1""",1
"""ENST00000262215.8""",1


In [8]:
a = mane_exons['gene_type'].value_counts(sort=True)
a

gene_type,count
str,u32
"""protein_coding""",480590


In [9]:
b = mane_exons['Feature'].value_counts(sort=True)
b

Feature,count
str,u32
"""exon""",202056
"""CDS""",191762
"""UTR""",48232
"""start_codon""",19259
"""stop_codon""",19245
"""Selenocysteine""",36


In [10]:
non_mane_exons = (
    gtf_exp
    .filter(
        (pl.col('gene_type') == 'protein_coding') |
        (pl.col('gene_type').str.contains('pseudogene')) |
        (pl.col('Feature') == 'CDS')
    )
    .filter(pl.col('Feature') != 'transcript')
    .filter(~pl.col('tag').str.contains('MANE_Select'))
)

non_mane_exons

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
u32,str,str,str,i64,i64,str,str,str,str,str,str,str,i32,str,str,str,str,i32,str,str,str,str,str,str,str,str,str
126,"""chr1""","""HAVANA""","""exon""",12010,12057,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",1,"""ENSE00001948541.1""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
127,"""chr1""","""HAVANA""","""exon""",12179,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",2,"""ENSE00001671638.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
128,"""chr1""","""HAVANA""","""exon""",12613,12697,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",3,"""ENSE00001758273.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
129,"""chr1""","""HAVANA""","""exon""",12975,13052,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",4,"""ENSE00001799933.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
130,"""chr1""","""HAVANA""","""exon""",13221,13374,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",5,"""ENSE00001746346.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7750138,"""chrM""","""ENSEMBL""","""CDS""",14149,14673,""".""","""-""","""0""","""gene_id ""ENSG00000198695.2""; t…","""ENSG00000198695.2""","""protein_coding""","""MT-ND6""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000361681.2""","""protein_coding""","""MT-ND6-201""",1,"""ENSE00001434974.2""","""NA""",null,"""HGNC:7462""",null,null,"""ENSP00000354665.2""",null,null
7750139,"""chrM""","""ENSEMBL""","""start_codon""",14671,14673,""".""","""-""","""0""","""gene_id ""ENSG00000198695.2""; t…","""ENSG00000198695.2""","""protein_coding""","""MT-ND6""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000361681.2""","""protein_coding""","""MT-ND6-201""",1,"""ENSE00001434974.2""","""NA""",null,"""HGNC:7462""",null,null,"""ENSP00000354665.2""",null,null
7750145,"""chrM""","""ENSEMBL""","""exon""",14747,15887,""".""","""+""",""".""","""gene_id ""ENSG00000198727.2""; t…","""ENSG00000198727.2""","""protein_coding""","""MT-CYB""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000361789.2""","""protein_coding""","""MT-CYB-201""",1,"""ENSE00001436074.2""","""NA""",null,"""HGNC:7427""",null,null,"""ENSP00000354554.2""",null,null


In [11]:
non_mane_exons[['gene_id', 'gene_name', 'transcript_id']].unique()['transcript_id'].value_counts(sort=True)

transcript_id,count
str,u32
null,17373
"""ENST00000941635.1""",1
"""ENST00000597853.5""",1
"""ENST00000375375.7""",1
"""ENST00000686988.1""",1
…,…
"""ENST00000457574.1""",1
"""ENST00000966798.1""",1
"""ENST00000967079.1""",1


In [12]:
non_mane_cds = non_mane_exons.filter(pl.col('Feature') == 'CDS')

non_mane_exonic = non_mane_exons.filter(pl.col('Feature') == 'exon')
non_mane_exonic

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
u32,str,str,str,i64,i64,str,str,str,str,str,str,str,i32,str,str,str,str,i32,str,str,str,str,str,str,str,str,str
126,"""chr1""","""HAVANA""","""exon""",12010,12057,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",1,"""ENSE00001948541.1""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
127,"""chr1""","""HAVANA""","""exon""",12179,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",2,"""ENSE00001671638.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
128,"""chr1""","""HAVANA""","""exon""",12613,12697,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",3,"""ENSE00001758273.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
129,"""chr1""","""HAVANA""","""exon""",12975,13052,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",4,"""ENSE00001799933.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
130,"""chr1""","""HAVANA""","""exon""",13221,13374,""".""","""+""",""".""","""gene_id ""ENSG00000223972.6""; t…","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""",2,"""basic, Ensembl_canonical, GENC…","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",5,"""ENSE00001746346.2""","""NA""","""OTTHUMT00000002844.2""""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000005, PGO:0000019""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7750110,"""chrM""","""ENSEMBL""","""exon""",10470,10766,""".""","""+""",""".""","""gene_id ""ENSG00000212907.2""; t…","""ENSG00000212907.2""","""protein_coding""","""MT-ND4L""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000361335.1""","""protein_coding""","""MT-ND4L-201""",1,"""ENSE00001596097.1""","""NA""",null,"""HGNC:7460""",null,null,"""ENSP00000354728.1""",null,null
7750117,"""chrM""","""ENSEMBL""","""exon""",10760,12137,""".""","""+""",""".""","""gene_id ""ENSG00000198886.2""; t…","""ENSG00000198886.2""","""protein_coding""","""MT-ND4""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000361381.2""","""protein_coding""","""MT-ND4-201""",1,"""ENSE00001666004.2""","""NA""",null,"""HGNC:7459""",null,null,"""ENSP00000354961.2""",null,null
7750131,"""chrM""","""ENSEMBL""","""exon""",12337,14148,""".""","""+""",""".""","""gene_id ""ENSG00000198786.2""; t…","""ENSG00000198786.2""","""protein_coding""","""MT-ND5""",3,"""basic, Ensembl_canonical, GENC…","""ENST00000361567.2""","""protein_coding""","""MT-ND5-201""",1,"""ENSE00001435330.2""","""NA""",null,"""HGNC:7461""",null,null,"""ENSP00000354813.2""",null,null


### Convert regions to pyranges

In [13]:
mane_exons_pr = pr.PyRanges(mane_exons.filter(pl.col('Feature') == 'exon').to_pandas()).merge()
mane_exons_pr

,Chromosome,Start,End,Strand
0,chr1,65419,65433,+
1,chr1,65520,65573,+
2,chr1,69037,71585,+
3,chr1,923923,924948,+
4,chr1,925922,926013,+
...,...,...,...,...
201727,chrY,25041769,25041886,-
201728,chrY,25043946,25044064,-
201729,chrY,25048474,25048610,-
201730,chrY,25051676,25051798,-


In [14]:
mane_cds_pr = pr.PyRanges(mane_exons.filter(pl.col('Feature') == 'CDS').to_pandas()).merge()
mane_utr_pr = pr.PyRanges(mane_exons.filter(pl.col('Feature') == 'UTR').to_pandas()).merge()

mane_cds_pr

,Chromosome,Start,End,Strand
0,chr1,65565,65573,+
1,chr1,69037,70005,+
2,chr1,924432,924948,+
3,chr1,925922,926013,+
4,chr1,930155,930336,+
...,...,...,...,...
191593,chrY,24813183,24813185,-
191594,chrY,25038101,25038116,-
191595,chrY,25038809,25038914,-
191596,chrY,25041769,25041886,-


In [15]:
non_mane_exons_pr = pr.PyRanges(non_mane_exons.filter(pl.col('Feature') == 'exon').to_pandas()).merge()
non_mane_exons_pr = non_mane_exons_pr.subtract(mane_exons_pr)
non_mane_exons_pr

,Chromosome,Start,End,Strand
0,chr1,12010,12057,+
1,chr1,12179,12227,+
2,chr1,12613,12697,+
3,chr1,12975,13052,+
4,chr1,13221,13374,+
...,...,...,...,...
125091,chrY,57212184,57213125,-
125092,chrY,57213204,57213357,-
125093,chrY,57213526,57213602,-
125094,chrY,57213880,57213964,-


In [16]:
non_mane_cds_pr = pr.PyRanges(non_mane_exons.filter(pl.col('Feature') == 'CDS').to_pandas()).merge()
non_mane_cds_pr = non_mane_cds_pr.subtract(mane_cds_pr)

non_mane_utr_pr = pr.PyRanges(non_mane_exons.filter(pl.col('Feature') == 'UTR').to_pandas()).merge()
non_mane_utr_pr = non_mane_utr_pr.subtract(mane_utr_pr)

non_mane_cds_pr

,Chromosome,Start,End,Strand
0,chr1,939272,939275,+
1,chr1,939412,939460,+
2,chr1,943808,943908,+
3,chr1,962286,962355,+
4,chr1,963032,963109,+
...,...,...,...,...
39577,chrY,18769601,18769619,-
39578,chrY,18770183,18770292,-
39579,chrY,19723341,19723433,-
39580,chrY,19741300,19741318,-


### Annotate UKBBGym variants

In [17]:
anno = (
    pl.read_parquet(
        '/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_genocode_251103.parquet', 
        columns=['id', 'chrom', 'pos', 'ref', 'alt', 'gene']
    )
    .with_columns(
        Chromosome = pl.col('chrom'),
        Start = pl.col('pos') - 1,
        End = pl.col('pos') - 1 + pl.col('ref').str.len_chars()
    )
    .select(['id', 'Chromosome', 'Start', 'End', 'ref', 'alt', 'gene'])
)

anno_pr = pr.PyRanges(anno.to_pandas())
anno_pr

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:214626910:A:G,chr1,214626909,214626910,A,G,ENSG00000117724
1,chr1:21556831:T:G,chr1,21556830,21556831,T,G,ENSG00000162551
2,chr1:21569280:C:A,chr1,21569279,21569280,C,A,ENSG00000162551
3,chr1:21556795:T:C,chr1,21556794,21556795,T,C,ENSG00000162551
4,chr1:214648630:A:G,chr1,214648629,214648630,A,G,ENSG00000117724
...,...,...,...,...,...,...,...
25373672,chr22:17093668:AGGTAAC:A,chr22,17093667,17093674,AGGTAAC,A,ENSG00000177663
25373673,chr22:36919903:GCATA:G,chr22,36919902,36919907,GCATA,G,ENSG00000100368
25373674,chr22:41102783:GT:G,chr22,41102782,41102784,GT,G,ENSG00000100393
25373675,chr22:24508167:ATC:A,chr22,24508166,24508169,ATC,A,ENSG00000100024


In [18]:
mane_exons_pr_vars = anno_pr.overlap(mane_exons_pr)#.df['gene'].value_counts()
mane_cds_pr_vars = anno_pr.overlap(mane_cds_pr)
mane_utr_pr_vars = anno_pr.overlap(mane_utr_pr)

mane_cds_pr_vars

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:214646777:A:G,chr1,214646776,214646777,A,G,ENSG00000117724
1,chr1:214655398:A:G,chr1,214655397,214655398,A,G,ENSG00000117724
2,chr1:214646100:C:A,chr1,214646099,214646100,C,A,ENSG00000117724
3,chr1:21554093:A:G,chr1,21554092,21554093,A,G,ENSG00000162551
4,chr1:214614850:G:A,chr1,214614849,214614850,G,A,ENSG00000117724
...,...,...,...,...,...,...,...
784685,chr22:50526097:G:GCAGCAC,chr22,50526096,50526097,G,GCAGCAC,ENSG00000025708
784686,chr22:20787014:G:GC,chr22,20787013,20787014,G,GC,ENSG00000099937
784687,chr22:36285713:T:TC,chr22,36285712,36285713,T,TC,ENSG00000100345
784688,chr22:28687932:TG:T,chr22,28687931,28687933,TG,T,ENSG00000183765


In [19]:
# Convert to polars and add MANE / non-MANE flags

non_mane_exons_pr_vars = anno_pr.overlap(non_mane_exons_pr)
non_mane_cds_pr_vars = anno_pr.overlap(non_mane_cds_pr)
non_mane_utr_pr_vars = anno_pr.overlap(non_mane_utr_pr)

non_mane_cds_pr_vars

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:198256393:C:T,chr1,198256392,198256393,C,T,ENSG00000151414
1,chr1:21547457:C:T,chr1,21547456,21547457,C,T,ENSG00000162551
2,chr1:21547499:T:G,chr1,21547498,21547499,T,G,ENSG00000162551
3,chr1:198277942:A:T,chr1,198277941,198277942,A,T,ENSG00000151414
4,chr1:198256375:A:T,chr1,198256374,198256375,A,T,ENSG00000151414
...,...,...,...,...,...,...,...
59856,chr22:40156181:CAGTG:C,chr22,40156180,40156185,CAGTG,C,ENSG00000100354
59857,chr22:37084218:A:AG,chr22,37084217,37084218,A,AG,ENSG00000187045
59858,chr22:40847996:G:GA,chr22,40847995,40847996,G,GA,ENSG00000100380
59859,chr22:24502475:CT:C,chr22,24502474,24502476,CT,C,ENSG00000100024


In [20]:
# Convert to polars and add MANE / non-MANE flags

mane_exons_pl = pl.from_pandas(mane_exons_pr_vars.df).with_columns(MANE_exonic = True).select(['id', 'gene', 'MANE_exonic'])
mane_cds_pl = pl.from_pandas(mane_cds_pr_vars.df).with_columns(MANE_CDS = True).select(['id', 'gene', 'MANE_CDS'])
mane_utr_pl = pl.from_pandas(mane_utr_pr_vars.df).with_columns(MANE_UTR = True).select(['id', 'gene', 'MANE_UTR'])

non_mane_exons_pl = pl.from_pandas(non_mane_exons_pr_vars.df).with_columns(non_MANE_exonic = True).select(['id', 'gene', 'non_MANE_exonic'])
non_mane_cds_pl = pl.from_pandas(non_mane_cds_pr_vars.df).with_columns(non_MANE_CDS = True).select(['id', 'gene', 'non_MANE_CDS'])
non_mane_utr_pl = pl.from_pandas(non_mane_utr_pr_vars.df).with_columns(non_MANE_UTR = True).select(['id', 'gene', 'non_MANE_UTR'])

non_mane_exons_pl

id,gene,non_MANE_exonic
str,str,bool
"""chr1:214652675:C:T""","""ENSG00000117724""",true
"""chr1:198256393:C:T""","""ENSG00000151414""",true
"""chr1:214608779:C:T""","""ENSG00000117724""",true
"""chr1:198221715:T:A""","""ENSG00000151414""",true
"""chr1:198198084:G:C""","""ENSG00000151414""",true
…,…,…
"""chr22:24495324:C:CGGGCCTGGGCGC…","""ENSG00000100024""",true
"""chr22:36325562:CCAA:C""","""ENSG00000100345""",true
"""chr22:17750219:T:TG""","""ENSG00000015475""",true


### Merge with annotations df

In [21]:
vep_CDS = [
    'consequence_coding_sequence_variant',
    'consequence_frameshift_variant',
    'consequence_inframe_deletion',
    'consequence_inframe_insertion',
    'consequence_missense_variant',
    'consequence_protein_altering_variant',
    'consequence_start_lost',
    'consequence_start_retained_variant',
    'consequence_stop_gained',
    'consequence_stop_lost',
    'consequence_stop_retained_variant',
    'consequence_synonymous_variant',
    'consequence_splice_acceptor_variant',
    'consequence_splice_donor_variant',
    # 'consequence_nmd_transcript_variant',
]

In [22]:
anno = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_genocode_251103.parquet').drop(['vep_coding', 'gencode_non_coding', 'vep_gencode_non_coding'])

anno

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,ensembleregulatoryfeature_is_nan,dbscsnv-ada_score_is_nan,dbscsnv-rf_score_is_nan,remapoverlaptf_is_nan,remapoverlapcl_is_nan,esmscoremissense_is_nan,esmscoreinframe_is_nan,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0,0.153293
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0,1.684965
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0,-1.212965
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0,-1.479978
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0,-2.904763
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0,null
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.0,0.095799,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,0,0,1,1,

In [23]:
# Merge all MANE and non-MANE annotations

mane_annos = (
    anno[['id', 'gene']]
    .join(mane_exons_pl, on=['id', 'gene'], how='left')
    .join(mane_cds_pl, on=['id', 'gene'], how='left')
    .join(mane_utr_pl, on=['id', 'gene'], how='left')
    .join(non_mane_exons_pl, on=['id', 'gene'], how='left')
    .join(non_mane_cds_pl, on=['id', 'gene'], how='left')
    .join(non_mane_utr_pl, on=['id', 'gene'], how='left')

    .fill_null(False)
)

mane_annos

id,gene,MANE_exonic,MANE_CDS,MANE_UTR,non_MANE_exonic,non_MANE_CDS,non_MANE_UTR
str,str,bool,bool,bool,bool,bool,bool
"""chr2:169197167:T:G""","""ENSG00000081479""",false,false,false,false,false,false
"""chr2:169204292:A:T""","""ENSG00000081479""",false,false,false,false,false,false
"""chr2:169030531:C:T""","""ENSG00000073734""",false,false,false,false,false,false
"""chr2:168910825:T:G""","""ENSG00000073734""",false,false,false,false,false,false
"""chr2:168969544:T:C""","""ENSG00000073734""",true,true,false,false,false,true
…,…,…,…,…,…,…,…
"""chr15:34974609:ACTTCT:A""","""ENSG00000198146""",false,false,false,false,false,false
"""chr9:132358570:C:A""","""ENSG00000107290""",false,false,false,false,false,false
"""chr8:13200986:C:CA""","""ENSG00000164741""",false,false,false,false,false,false


In [24]:
final_annos = (
    anno
    .join(mane_annos, on=['id', 'gene'], how='left')
    .with_columns(
        VEP_CDS = pl.sum_horizontal(vep_CDS) > 0,
    )
)

final_annos.write_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_MANE_251104.parquet')
final_annos

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean,MANE_exonic,MANE_CDS,MANE_UTR,non_MANE_exonic,non_MANE_CDS,non_MANE_UTR,VEP_CDS
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64,bool,bool,bool,bool,bool,bool,bool
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0,0.153293,false,false,false,false,false,false,false
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0,1.684965,false,false,false,false,false,false,false
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0,-1.212965,false,false,false,false,false,false,false
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0,-1.479978,false,false,false,false,false,false,false
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0,-2.904763,true,true,false,false,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0,null,false,false,false,false,false,false,false
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.

## Debugging

In [2]:
final_annos = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_MANE_251104.parquet')
final_annos

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean,MANE_exonic,MANE_CDS,MANE_UTR,non_MANE_exonic,non_MANE_CDS,non_MANE_UTR,VEP_CDS
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64,bool,bool,bool,bool,bool,bool,bool
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0,0.153293,false,false,false,false,false,false,false
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0,1.684965,false,false,false,false,false,false,false
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0,-1.212965,false,false,false,false,false,false,false
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0,-1.479978,false,false,false,false,false,false,false
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0,-2.904763,true,true,false,false,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0,null,false,false,false,false,false,false,false
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.

In [3]:
final_annos.filter(
    pl.col('VEP_CDS') == (pl.col('MANE_CDS')) # | pl.col('non_MANE_CDS'))
)

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean,MANE_exonic,MANE_CDS,MANE_UTR,non_MANE_exonic,non_MANE_CDS,non_MANE_UTR,VEP_CDS
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64,bool,bool,bool,bool,bool,bool,bool
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0,0.153293,false,false,false,false,false,false,false
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0,1.684965,false,false,false,false,false,false,false
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0,-1.212965,false,false,false,false,false,false,false
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0,-1.479978,false,false,false,false,false,false,false
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0,-2.904763,true,true,false,false,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0,null,false,false,false,false,false,false,false
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.

In [5]:
final_annos[['MANE_exonic', 'non_MANE_exonic', 'MANE_CDS', 'non_MANE_CDS', 'MANE_UTR', 'non_MANE_UTR', 'VEP_CDS']].describe()

statistic,MANE_exonic,non_MANE_exonic,MANE_CDS,non_MANE_CDS,MANE_UTR,non_MANE_UTR,VEP_CDS
str,f64,f64,f64,f64,f64,f64,f64
"""count""",2.5373677e7,2.5373677e7,2.5373677e7,2.5373677e7,2.5373677e7,2.5373677e7,2.5373677e7
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.059322,0.027294,0.030925,0.002359,0.028491,0.022531,0.027117
"""std""",null,null,null,null,null,null,null
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null
"""max""",1.0,1.0,1.0,1.0,1.0,1.0,1.0
